# 연습용 실험 기록을 남겨 공식 실행과 구분하기

## 이번 질문

이 노트북은 시간이 남을 때 여는 선택 실습입니다. 본편에서 후보 보류/승인을 이미 기록한 뒤에만 실행합니다. 1장 선택 노트북에서 학습/검증 지문을 확인한 뒤에 엽니다.

학습 데이터와 검증 데이터만 사용해 모델을 한 번 맞추고, 그 결과를 클러스터 MLflow에 남깁니다. 기록 위치는 환경 변수 `AIQA_MLFLOW_TRACKING_URI`입니다. 이 실행 번호는 공식 평가나 배포 근거가 아닙니다. 공식 실행 번호는 선언 JSON에서 읽어 비교만 합니다.

## 먼저 예상

클러스터에 남긴 연습 실행 번호가 공식 Candidate B 실행 번호와 같을 수 있는지 한 문장으로 적습니다. MLflow 화면이 비어 있어도 공식 판단이 가능한지도 예상합니다.

## 실행과 관측

### 1. 개발용 데이터만 읽고 클러스터 기록 위치 확인하기

학습과 평가는 `train`과 `valid`만 사용합니다. 기록 위치는 강사가 준 `AIQA_MLFLOW_TRACKING_URI`입니다. localhost 기본값, 임시 sqlite, ClusterIP, port-forward는 쓰지 않습니다. 저장소의 `artifacts/mlflow/`나 공식 근거 폴더에는 쓰지 않습니다. 값이 없거나 `/health`가 실패하면 `MLFLOW_NOT_RUNNING`만 남기고 실행을 만들지 않습니다. 노트북이 끝나면 원래 기록 위치를 되돌립니다.

In [ ]:
import json
import os
import urllib.error
import urllib.request
from pathlib import Path

import mlflow
import pandas as pd

ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir()
)
features_path = ROOT / "data/processed/physionet-2012/patient-features.csv"
split_path = ROOT / "data/splits/physionet-2012/revisions/v2/split-manifest.csv"
manifest_path = ROOT / (
    "docs/reference/evidence/model/revisions/v2/release-manifest.json"
)

# 공식 실행 번호는 JSON에 이미 있습니다. 여기서 새로 만들지 않습니다.
release = json.loads(manifest_path.read_text(encoding="utf-8"))
official_train_run = release["approved_model"]["model_mlflow_run_id"]
official_final_run = release["approved_model"]["final_mlflow_run_id"]

features = pd.read_csv(features_path)
splits = pd.read_csv(split_path)
# 학습/검증 행만 남깁니다.
development = (
    features.merge(splits, on="record_id", validate="one_to_one")
    .loc[lambda frame: frame["role"].isin(["train", "valid"])]
    .copy()
)
train = development.loc[development["role"].eq("train")].copy()
valid = development.loc[development["role"].eq("valid")].copy()
feature_columns = [
    column
    for column in development.columns
    if column not in {"record_id", "target", "role"}
]

# 노트북이 공유 환경의 다음 실행을 바꾸지 않도록 이전 위치를 기억합니다.
previous_tracking_uri = mlflow.get_tracking_uri()
tracking_uri = os.environ.get("AIQA_MLFLOW_TRACKING_URI", "").strip()


def mlflow_health_ok(uri: str) -> bool:
    """클러스터 Ingress /health가 응답하면 True입니다."""
    if not uri.startswith(("http://", "https://")):
        return False
    try:
        with urllib.request.urlopen(f"{uri.rstrip('/')}/health", timeout=3) as response:
            return 200 <= int(response.status) < 300
    except (urllib.error.URLError, TimeoutError, ValueError, OSError):
        return False


tracking_ready = bool(tracking_uri) and mlflow_health_ok(tracking_uri)
if not tracking_ready:
    tracking_status = pd.DataFrame(
        [
            {
                "status": "MLFLOW_NOT_RUNNING",
                "tracking_uri": tracking_uri or "(unset)",
                "next_action": (
                    "강사가 준 AIQA_MLFLOW_TRACKING_URI를 설정하고 "
                    "클러스터 MLflow Ingress /health가 응답하는지 확인합니다."
                ),
            }
        ]
    )
else:
    mlflow.set_tracking_uri(tracking_uri)
    experiment = mlflow.set_experiment("practice-development-tracking")
    tracking_status = pd.DataFrame(
        [
            {
                "status": "MLFLOW_READY",
                "experiment": experiment.name,
                "train_rows": len(train),
                "valid_rows": len(valid),
                "official_train_run": official_train_run,
                "official_final_run": official_final_run,
            }
        ]
    )
tracking_status


### 2. 한 번 학습하고 연습 실행으로 기록하기

모델은 학습 데이터에만 맞추고, 지표는 검증 데이터에서 계산합니다. 클러스터 MLflow가 준비된 때만 설정값과 지표를 연습 실행에 남긴 뒤 같은 저장 위치에서 다시 조회합니다. 준비되지 않았으면 실행을 만들지 않습니다.

이 숫자는 개발용 재현입니다. 공식 후보 평가를 대체하지 않습니다.

In [ ]:
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

practice_run_id = None
logged_runs = []
tp = fp = fn = tn = 0
if tracking_ready:
    threshold = 0.50
    model = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(max_iter=2000, random_state=42)),
        ]
    )

    with mlflow.start_run(run_name="practice-train-valid") as active_run:
        # 학습 데이터로만 맞춥니다.
        model.fit(train[feature_columns], train["target"])
        scores = model.predict_proba(valid[feature_columns])[:, 1]
        predictions = (scores >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(
            valid["target"], predictions, labels=[0, 1]
        ).ravel()
        practice_metrics = {
            "valid_precision": float(
                precision_score(valid["target"], predictions, zero_division=0)
            ),
            "valid_recall": float(
                recall_score(valid["target"], predictions, zero_division=0)
            ),
            "valid_pr_auc": float(average_precision_score(valid["target"], scores)),
            "valid_fn": float(fn),
            "valid_fp": float(fp),
        }
        mlflow.log_params({"threshold": threshold, "max_iter": 2000, "random_state": 42})
        mlflow.log_metrics(practice_metrics)
        mlflow.set_tags(
            {
                "purpose": "leftover-development-practice",
                "data_roles": "train,valid",
                "not_official_evidence": "true",
            }
        )
        practice_run_id = active_run.info.run_id

    # 방금 남긴 실행이 조회되는지 확인합니다.
    logged_runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
{
    "practice_run_id": practice_run_id,
    "official_train_run": official_train_run,
    "official_final_run": official_final_run,
    "logged_run_count": int(len(logged_runs)),
}


## 해석과 기록

### 3. 내 실행과 공식 실행을 섞지 않기

방금 출력된 실행 번호는 이 노트북이 만든 연습 기록입니다. 공식 학습 실행과 공식 승인 실행은 JSON에 이미 고정돼 있습니다. 화면이 비어 있어도 그 번호는 바뀌지 않습니다.

판단 기록의 모델 품질 칸에는 공식 JSON 경로와 공식 실행 번호를 유지합니다. 이 노트북의 실행 번호를 승인 근거로 바꾸지 않습니다.

## 결과 점검

아래 검사는 개발용 역할만 사용했는지, 연습 실행이 공식 번호와 다른지, 기록 위치를 되돌렸는지 확인합니다.

In [ ]:
accessed_roles = set(development["role"].unique())
assert accessed_roles == {"train", "valid"}
assert len(train) == 2900
assert len(valid) == 600
if tracking_ready:
    assert practice_run_id != official_train_run
    assert practice_run_id != official_final_run
    assert int(len(logged_runs)) == 1
    assert int(tp + fp + fn + tn) == len(valid)
else:
    assert practice_run_id is None

# 공유 커널의 다음 실습이 이 위치를 쓰지 않게 되돌립니다.
mlflow.set_tracking_uri(previous_tracking_uri)
prefix = (
    "클러스터 MLflow 연습 실행을 남겼습니다. "
    if tracking_ready
    else "클러스터 MLflow가 없어 연습 실행을 만들지 않았습니다. "
)
print(prefix + "공식 실행 번호와 근거 파일은 변경하지 않았습니다.")


## 다음 확인

본편으로 돌아가 공식 JSON의 실행 번호를 읽습니다. 이 노트북의 연습 실행을 그 번호 대신 쓰지 않습니다. 화면은 공식 번호를 눈으로 확인하는 선택 자료이며, 화면이 없어도 JSON으로 본편을 닫습니다.